# Deep Ensemble with DNABERT

In [1]:
# Run once if not already installed
!pip install torch transformers peft accelerate

In [12]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader, TensorDataset, Dataset

## Data Loading

In [4]:
DATA_DIR = 'data/processed/'

# Load dataframes with raw sequences for DNABERT tokenisation
guide_train = pd.read_csv(DATA_DIR + 'guide_train.csv', low_memory=False)
guide_test = pd.read_csv(DATA_DIR + 'guide_test.csv', low_memory=False)
change_train = pd.read_csv(DATA_DIR + 'change_train.csv', low_memory=False)
change_test = pd.read_csv(DATA_DIR + 'change_test.csv', low_memory=False)

print(f'GUIDE-seq train: {guide_train.shape}, test: {guide_test.shape}')
print(f'CHANGE-seq train: {change_train.shape}, test: {change_test.shape}')

GUIDE-seq train: (1229372, 12), test: (248631, 12)
CHANGE-seq train: (2352636, 10), test: (520991, 10)


## DNABERT Tokenisation

In [6]:
GRNA_COL = 'target'
TARGET_COL = 'offtarget_sequence'
DNABERT_MODEL = 'zhihan1996/DNA_bert_3'

# Load tokeniser
tokenizer = AutoTokenizer.from_pretrained(
    DNABERT_MODEL,
    trust_remote_code=True
)
print(f'Tokeniser loaded: {DNABERT_MODEL}')

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_3:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Tokeniser loaded: zhihan1996/DNA_bert_3


In [10]:
# K = 3

# def sequence_to_kmers(seq, k=K):
#     return ' '.join(seq[i:i+k] for i in range(len(seq) - k + 1))

# def tokenise_sequences(df, grna_col, target_col, tokenizer):
#     # Convert each sequence to 3-mers independently to avoid boundary k-mers
#     grna_kmers = df[grna_col].apply(sequence_to_kmers).tolist()
#     target_kmers = df[target_col].apply(sequence_to_kmers).tolist()
    
#     encoded = tokenizer(
#         grna_kmers,
#         target_kmers,
#         padding=True,
#         truncation=True,
#         return_tensors='pt'
#     )
#     return encoded

# print('Tokenising GUIDE-seq train...')
# guide_train_encoded = tokenise_sequences(guide_train, GRNA_COL, TARGET_COL, tokenizer)
# print('Done')
# print(f'Input IDs shape: {guide_train_encoded["input_ids"].shape}')

## Note on Tokenisation
The `tokenise_sequences` function above was verified to work correctly with `DNA_bert_3`, producing input IDs of shape `[n_samples, 45]`. Full tokenisation of the training set is not run here to avoid memory issues with 1.2M+ samples. Tokenisation is handled on-the-fly in the Dataset class below.

In [13]:
class CRISPRDataset(Dataset):
    def __init__(self, df, grna_col, target_col, tokenizer, max_length=45):
        self.df = df.reset_index(drop=True)
        self.grna_col = grna_col
        self.target_col = target_col
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = torch.tensor(
            self.df['label'].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Convert sequences to 3-mers on-the-fly
        grna_kmers = sequence_to_kmers(self.df[self.grna_col].iloc[idx])
        target_kmers = sequence_to_kmers(self.df[self.target_col].iloc[idx])

        encoded = self.tokenizer(
            grna_kmers,
            target_kmers,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'token_type_ids': encoded['token_type_ids'].squeeze(0),
            'label': self.labels[idx]
        }

# Quick check
dataset = CRISPRDataset(guide_train, GRNA_COL, TARGET_COL, tokenizer)
print(f'Dataset size: {len(dataset)}')
sample = dataset[0]
print(f'input_ids shape: {sample["input_ids"].shape}')
print(f'attention_mask shape: {sample["attention_mask"].shape}')
print(f'token_type_ids shape: {sample["token_type_ids"].shape}')
print(f'label: {sample["label"]}')

Dataset size: 1229372
input_ids shape: torch.Size([45])
attention_mask shape: torch.Size([45])
token_type_ids shape: torch.Size([45])
label: 1.0


## LoRA Configuration and Model

In [19]:
# Load DNABERT with a custom classification head for binary off-target prediction
class DNABERTClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        self.dropout = nn.Dropout(0.1)
        # Map CLS token embedding to single logit for binary classification
        self.classifier = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_embedding))
        return logits

# LoRA config - parameter-efficient adaptation to reduce
# trainable parameters and memory requirements
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none'
)

def build_lora_model():
    # Fresh backbone load for each call - ensures independent LoRA initialisations
    # across ensemble members
    model = DNABERTClassifier(DNABERT_MODEL)
    model.backbone = get_peft_model(model.backbone, lora_config)
    return model

# Test build
model = build_lora_model()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_3
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Check trainable parameters - backbone LoRA only vs full model
model.backbone.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Total trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)')
print(f'Classifier trainable: {model.classifier.weight.requires_grad}')

trainable params: 294,912 || all params: 86,389,248 || trainable%: 0.3414
Total trainable: 295,681 / 86,390,017 (0.34%)
Classifier trainable: True
